# Exaone 4.0 버전이 7월에 나왔다고 해서 써 봤는데, 학원 시스템에서는 3.5가 더 나은 듯

### <span style="color: rgb(46, 204, 113);">1. 환경 설정 및 패키지 설치</span>
- 가상환경 만든 뒤, pip install ipykernel 실행
- 아래의 패키지 설치

In [15]:
%pip install -q langchain-community transformers accelerate bitsandbytes safetensors hf_xet

Note: you may need to restart the kernel to use updated packages.


In [16]:
# 1) PyTorch (CUDA 11.8 빌드)
%pip install --upgrade torch --index-url https://download.pytorch.org/whl/cu118

Looking in indexes: https://download.pytorch.org/whl/cu118
Note: you may need to restart the kernel to use updated packages.


In [5]:
# 2) 노트북에서 뜨는 tqdm 경고 해결 (선택)
%pip install ipywidgets


  Using cached ipywidgets-8.1.7-py3-none-any.whl.metadata (2.4 kB)
  Using cached widgetsnbextension-4.0.14-py3-none-any.whl.metadata (1.6 kB)
  Using cached jupyterlab_widgets-3.0.15-py3-none-any.whl.metadata (20 kB)
Using cached ipywidgets-8.1.7-py3-none-any.whl (139 kB)
Using cached jupyterlab_widgets-3.0.15-py3-none-any.whl (216 kB)
Using cached widgetsnbextension-4.0.14-py3-none-any.whl (2.2 MB)

   ------------- -------------------------- 1/3 [jupyterlab_widgets]
   -------------------------- ------------- 2/3 [ipywidgets]
   ---------------------------------------- 3/3 [ipywidgets]



In [15]:
content = """[전문개정 2009. 12. 31.]

[시행일: 2025. 1. 1.]  제15조제2호
제2관 소득의 종류와 금액 <개정 2009. 12. 31.>
제16조(이자소득)",./data/with_markdown-sample.docx,"이자소득 범위: 국가내국외국 채권, 국내 예금 이자 등 다양한 금융상품에서 발생하는 이익 포함.",이자배당,chunk_016,15
"4. 「상호저축은행법」에 따른 신용계(信用契) 또는 신용부금으로 인한 이익
5. 외국법인의 국내지점 또는 국내영업소에서 발행한 채권 또는 증권의 이자와 할인액
6. 외국법인이 ",./data/with_markdown-sample.docx,"이자 및 배당소득 범위 이자소득: 상호저축은행 신용계, 외국법인 채권증권 이자, 국외 예금",납세의무,chunk_017,5
"7. 「국제조세조정에 관한 법률」 제27조에 따라 배당받은 것으로 간주된 금액
8. 제43조에 따른 공동사업에서 발생한 소득금액 중 같은 조 제1항에 따른 출자공동사업자의 손익분",./data/with_markdown-sample.docx,"의제배당 및 자본전입 관련 규정: 주식 소각, 자본전입, 해산 잔여재산 분배 등으로 인한 초과 이익 배당",납세의무,chunk_018,27
5. 법인이 자기주식 또는 자기출자지분을 보유한 상태에서 제2호 각 목에 따른 자본전입을 함에 따라 그 법인 외의 주주 등의 지분비율이 증가한 경우 증가한 지분비율에 상당하는 주식
"""

In [4]:
from langchain_community.chat_models import ChatOllama
def extract_title_with_exaone3_5(content):
  """Exaone을 사용해서 제목 추출"""
  title_extractor_llm = ChatOllama(
                model="exaone3.5:2.4b",
                temperature=0.1, # 0에 가까우면 답변이 일관적
                num_predict=30 # 최대 토근 30토큰까지만 출력
            )
  prompt = f'''다음 content 내용의 핵심 제목을 30토큰 이내의 한줄로 간단히 완벽하게 말이 되도록 추출해 주세요. 중간에 말이 끊기면 안 되요.
  예 : "소득세 납세의무 범위 : 공동사업자별 소득과세, 상속과세, 증여자"
  content : {content}'''
  ai_message = title_extractor_llm.invoke(prompt)
  title = ai_message.content.strip()
  return title

print(extract_title_with_exaone3_5(content))

2025년 1월 1일 시행, 이자소득 및 다양 금융상품 이익 포함 과세 범위 확대: 상호저축은행 이자,


In [18]:
from transformers import pipeline, AutoTokenizer

model_id = "LGAI-EXAONE/EXAONE-4.0-1.2B"

# 토크나이저 먼저 받아와서 eos/pad 확인(토크나이저는 문장을 모델이 이해할 수 있는 숫자들의 나욜(token)로 바꿔주는 번역기)
tok = AutoTokenizer.from_pretrained(
    model_id, trust_remote_code=True # 이 모델은 허깅페이스에 있는 기본 코드 외에 별도의 코드를 함께 다운로드해야 동작하기 때문에, 해당 코드를 신뢰하고 실행하겠다는 의미로 이 옵션을 켜줍니다.
    )
eos_id = tok.eos_token_id
pad_id = tok.pad_token_id if tok.pad_token_id is not None else eos_id

# 1) pipeline 초기화 (device_map 권장)
exa4 = pipeline(
    "text-generation",
    model=model_id,
    tokenizer=tok,                # 같은 토크나이저 사용
    device_map="auto",            # device=0 대신 이거 추천
    torch_dtype="auto",
    trust_remote_code=True
)

def extract_title_with_exaone4(content: str) -> str:
    # 2) chat 템플릿으로 프롬프트 렌더링
    #    (add_generation_prompt=True가 포인트)
    messages = [
        {
            "role": "user",
            "content": (
                "다음 content 내용의 핵심 제목을 30토큰 이내의 한 줄로 간단히, "
                "완벽하게 문장이 끊기지 않도록 추출해 주세요.\n"
                '예: "소득세 납세의무 범위 : 공동사업자별 소득과세, 상속과세, 증여자"\n'
                f"content:\n{content}"
            )
        }
    ]
    prompt_str = tok.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True # "이제 네(모델)가 대답할 차례야" 라는 신호를 프롬프트 끝에 붙여주는 역할
    )

    # 3) eos/pad 지정 + 생성
    out = exa4(
        prompt_str,
        max_new_tokens=30,           # 20→30 권장
        do_sample=False,             # temperature 무시됨 (의도대로)
        eos_token_id=eos_id,
        pad_token_id=pad_id,
        return_full_text=False       # 생성 부분만
    )

    text = out[0]["generated_text"].strip()
    # 한 줄로 정리
    return " ".join(text.split())

Device set to use cpu


In [19]:
extract_title_with_exaone4(content)

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


'"이자소득 범위: 국가내국외국 채권, 국내 예금 이자, 상호저축은행 신용계 이익, 외국법인 채권 이자, 국외 예금,'